<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-00-prerequisites/lesson-0.1-python-for-genai/notebooks/exercises-0_1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐍 Lesson 0.1: Python for GenAI — Practice Exercises

**Netsetos GenAI Course** | Module 0

6 exercises covering async, generators, Pydantic, error handling, decorators, and the production LLM call pattern.

**Research-enhanced:** Covers the Python skills that 80% of GenAI courses skip — the exact patterns used in production LLM apps.

---

## Exercise 1 (Easy): Async Speedup — 10 Parallel Calls

### 📚 Theory
LLM API calls are I/O-bound — 95% waiting for network. `asyncio.gather()` fires all calls simultaneously. 10 calls × 2s each: sequential = 20s, async = 2s.

### 📋 Task
1. `async def call_llm(prompt)` with `await asyncio.sleep(2)`
2. 10 prompts, measure sequential vs parallel time
3. Print the speedup ratio

In [ ]:
# YOUR CODE HERE
import asyncio, time

async def call_llm(prompt, delay=2.0):
    # TODO: await the delay, return a response string
    pass

async def main():
    prompts = [f'Q{i}' for i in range(10)]
    # TODO: sequential loop, then parallel with gather
    # TODO: print speedup ratio

await main()

<details><summary>✅ Solution</summary>



In [ ]:
import asyncio, time

async def call_llm(prompt, delay=2.0):
    await asyncio.sleep(delay)
    return f'Response: {prompt}'

async def main():
    prompts = [f'Q{i}' for i in range(10)]

    # Sequential
    start = time.time()
    for p in prompts:
        await call_llm(p)
    seq_time = time.time() - start

    # Parallel
    start = time.time()
    await asyncio.gather(*[call_llm(p) for p in prompts])
    par_time = time.time() - start

    print(f'Sequential: {seq_time:.1f}s | Async: {par_time:.1f}s | Speedup: {seq_time/par_time:.1f}x')

await main()

</details>

---

## Exercise 2 (Easy): Streaming Simulator

### 📚 Theory
Generators + `yield` display tokens as they arrive. TTFT drops from 2000ms to ~150ms. Every chat app uses this.

### 📋 Task
1. `stream_chars(text, delay=0.05)` — yield individual characters
2. `stream_words(text, delay=0.1)` — yield individual words
3. Use `flush=True` to see the streaming effect

In [ ]:
# YOUR CODE HERE
import time

def stream_chars(text, delay=0.05):
    # TODO: loop through chars, sleep, yield each
    pass

def stream_words(text, delay=0.1):
    # TODO: split into words, sleep, yield each word + space
    pass

print('Character streaming:')
for c in stream_chars('Hello!'):
    print(c, end='', flush=True)
print()

print('\nWord streaming:')
for w in stream_words('RAG combines retrieval with generation'):
    print(w, end='', flush=True)

<details><summary>✅ Solution</summary>



In [ ]:
import time

def stream_chars(text, delay=0.05):
    for c in text:
        time.sleep(delay)
        yield c

def stream_words(text, delay=0.1):
    for w in text.split():
        time.sleep(delay)
        yield w + ' '

print('Character streaming:')
for c in stream_chars('Hello from Gemini!'):
    print(c, end='', flush=True)
print()

print('\nWord streaming:')
for w in stream_words('RAG combines retrieval with generation'):
    print(w, end='', flush=True)

</details>

---

## Exercise 3 (Medium): Pydantic Model Library

### 📚 Theory
Pydantic is the #1 Python library for GenAI. It validates types AND constraints. OpenAI, LangChain, FastAPI all use it internally. When LLM returns `{"rating": "high"}` instead of `{"rating": 8}`, Pydantic catches it.

### 📋 Task
1. **ChatMessage**: role (user/assistant/system), content (1-10000), optional tokens
2. **CourseRecommendation**: name, score (0-1), difficulty, hours, topics
3. **CodeReview**: file, score (1-10), issues list
4. Test valid + invalid data

In [ ]:
# YOUR CODE HERE
from pydantic import BaseModel, Field
from typing import Optional

# TODO: Define ChatMessage, CourseRecommendation, CodeReview models
# TODO: Test with valid data
# TODO: Test with invalid data (should raise ValidationError)

<details><summary>✅ Solution</summary>



In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class ChatMessage(BaseModel):
    role: str = Field(..., pattern='^(user|assistant|system)$')
    content: str = Field(..., min_length=1, max_length=10000)
    tokens: Optional[int] = Field(None, ge=0)

class CourseRecommendation(BaseModel):
    course_name: str = Field(..., min_length=3)
    relevance_score: float = Field(..., ge=0.0, le=1.0)
    difficulty: str = Field(..., pattern='^(beginner|intermediate|advanced)$')
    estimated_hours: int = Field(..., ge=1, le=500)
    topics: list[str] = Field(..., min_length=1)

class CodeReview(BaseModel):
    file_name: str
    overall_score: int = Field(..., ge=1, le=10)
    issues: list[str] = Field(default_factory=list)

# --- Test valid data ---
msg = ChatMessage(role='assistant', content='Hello!', tokens=5)
print(f'✅ Valid ChatMessage: {msg.role} — {msg.content}')

course = CourseRecommendation(
    course_name='GenAI Engineering',
    relevance_score=0.95,
    difficulty='advanced',
    estimated_hours=100,
    topics=['LLM', 'RAG', 'agents']
)
print(f'✅ Valid CourseRecommendation: {course.course_name} ({course.relevance_score})')

# --- Test invalid data ---
try:
    ChatMessage(role='hacker', content='test')
except Exception as e:
    print(f'❌ Caught bad role: {type(e).__name__}')

try:
    CourseRecommendation(
        course_name='AI', relevance_score=1.5,
        difficulty='expert', estimated_hours=0, topics=[]
    )
except Exception as e:
    print(f'❌ Caught bad recommendation: {type(e).__name__}')

</details>

---

## Exercise 4 (Medium): Error Handling with Fallback Chain

### 📚 Theory
LLM APIs fail: 429 (rate limit) → wait and retry. 500 (server error) → retry. 401 (auth) → FAIL FAST. The production pattern: exponential backoff with jitter + model fallback chain (primary → cheaper → cached).

### 📋 Task
1. Create `RateLimitError`, `ServerError`, `AuthError` exceptions
2. `retry_with_backoff()` — exponential backoff with jitter, fail fast on auth
3. `call_with_fallback()` — try 3 models in sequence
4. Test with 40% failure rate

In [ ]:
# YOUR CODE HERE
import time, random

class RateLimitError(Exception): pass
class ServerError(Exception): pass
class AuthError(Exception): pass

# TODO: call_unreliable() — simulates 40% failure
# TODO: retry_with_backoff() — exponential backoff + jitter
# TODO: call_with_fallback() — primary → cheaper → cached

<details><summary>✅ Solution</summary>



In [ ]:
import time, random

class RateLimitError(Exception): pass
class ServerError(Exception): pass
class AuthError(Exception): pass

def call_unreliable(prompt):
    """Simulate an unreliable LLM API (40% failure rate)."""
    r = random.random()
    if r < 0.3:
        raise RateLimitError('429: Too Many Requests')
    if r < 0.4:
        raise ServerError('500: Internal Server Error')
    return f'Answer: {prompt}'

def retry_with_backoff(func, prompt, retries=3):
    """Retry with exponential backoff + jitter. Fail fast on AuthError."""
    for attempt in range(retries):
        try:
            return func(prompt)
        except AuthError:
            print('  Auth fail — STOP (not retrying)')
            raise
        except RateLimitError:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f'  Rate limited, waiting {wait:.1f}s')
            time.sleep(wait * 0.01)  # Scaled down for testing
        except ServerError:
            print(f'  Server error, retry {attempt + 1}')
            time.sleep(0.05)
    raise Exception(f'All {retries} retries failed')

def call_with_fallback(prompt):
    """Try models in order: primary → cheaper → cached."""
    for model in ['primary', 'cheaper', 'cached']:
        try:
            print(f'  Trying {model}...')
            if model == 'cached':
                return f'[Cached] {prompt}'
            return retry_with_backoff(call_unreliable, prompt)
        except Exception as e:
            print(f'  {model} failed: {e}')
    return 'Unavailable'

# Test
random.seed(42)
for i in range(5):
    result = call_with_fallback(f'GenAI concept {i}')
    print(f'  Result: {result}\n')

</details>

---

## Exercise 5 (Hard): Decorator Stack — @cache + @retry + @timer

### 📚 Theory
Production LLM calls stack decorators: `@timer @retry @cache @func`. Order matters. Cache is checked FIRST (innermost), retry wraps it, timer measures total. A cached call saves ~₹0.08. At 100K calls/day caching 40% = ₹3,200/day.

### 📋 Task
1. `@cache_response` — dict cache, print CACHE HIT
2. `@retry(max_attempts=3, delay=0.5)` — exponential backoff
3. `@timer` — print execution time in ms
4. Stack all three, test with 30% failure rate
5. Call twice with same prompt — second should be instant

In [ ]:
# YOUR CODE HERE
import time, functools, random

# TODO: cache_response decorator
# TODO: retry(max_attempts, delay) decorator
# TODO: timer decorator
# TODO: stack on call_llm, test twice with same prompt

<details><summary>✅ Solution</summary>



In [ ]:
import time, functools, random

def cache_response(func):
    """Cache results by arguments."""
    _cache = {}
    @functools.wraps(func)
    def wrapper(*args):
        key = str(args)
        if key in _cache:
            print('  CACHE HIT')
            return _cache[key]
        result = func(*args)
        _cache[key] = result
        return result
    return wrapper

def retry(max_attempts=3, delay=0.5):
    """Retry with exponential backoff."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args):
            for attempt in range(max_attempts):
                try:
                    return func(*args)
                except Exception:
                    if attempt < max_attempts - 1:
                        wait = delay * (2 ** attempt)
                        time.sleep(wait)
                    else:
                        raise
        return wrapper
    return decorator

def timer(func):
    """Print execution time in milliseconds."""
    @functools.wraps(func)
    def wrapper(*args):
        start = time.time()
        result = func(*args)
        elapsed_ms = (time.time() - start) * 1000
        print(f'  {elapsed_ms:.0f}ms')
        return result
    return wrapper

@timer
@retry(max_attempts=3, delay=0.3)
@cache_response
def call_llm(prompt):
    if random.random() < 0.3:
        raise ConnectionError('API timeout')
    time.sleep(0.5)
    return f'Answer: {prompt}'

print('Call 1 (new):')
print(call_llm('What is RAG?'))
print('\nCall 2 (cached):')
print(call_llm('What is RAG?'))

</details>

---

## Exercise 6 (Hard): Production LLM Pipeline — ALL Skills Combined

### 📚 Theory
This combines ALL 7 skills into the exact architecture used in Modules 5-11: Pydantic schema → async parallel calls → retry on failure → cache duplicates → validate response → stream output.

### 📋 Task
1. Pydantic: `LLMResponse(answer, confidence, tokens)`
2. Async `@cache` decorator
3. Async `call_llm()` with retry + Pydantic validation
4. Generator `stream()` for word-by-word output
5. Run 4 prompts (including duplicate) in parallel

In [ ]:
# YOUR CODE HERE
import asyncio, time, functools, random
from pydantic import BaseModel, Field

# TODO: LLMResponse Pydantic model
# TODO: async cache decorator
# TODO: async call_llm with retry + validation
# TODO: stream() generator
# TODO: main() — gather 4 prompts, stream results

<details><summary>✅ Solution</summary>



In [ ]:
import asyncio, time, functools, random
from pydantic import BaseModel, Field

# --- Pydantic model ---
class LLMResponse(BaseModel):
    answer: str = Field(..., min_length=1)
    confidence: float = Field(..., ge=0, le=1)
    tokens: int = Field(..., ge=1)

# --- Async cache decorator ---
def cache(func):
    _cache = {}
    @functools.wraps(func)
    async def wrapper(*args):
        key = str(args)
        if key in _cache:
            print(f'  CACHE: {args[0]}')
            return _cache[key]
        result = await func(*args)
        _cache[key] = result
        return result
    wrapper.cache = _cache
    return wrapper

# --- Async LLM call with retry + validation ---
@cache
async def call_llm(prompt):
    for attempt in range(3):
        try:
            await asyncio.sleep(random.uniform(0.3, 0.8))
            if random.random() < 0.2:
                raise ConnectionError('timeout')
            return LLMResponse(
                answer=f'GenAI explanation for {prompt}',
                confidence=round(random.uniform(0.7, 0.99), 2),
                tokens=random.randint(30, 150)
            )
        except ConnectionError:
            if attempt < 2:
                await asyncio.sleep(2 ** attempt * 0.1)
            else:
                raise

# --- Streaming generator ---
def stream(text, delay=0.02):
    for word in text.split():
        time.sleep(delay)
        yield word + ' '

# --- Main pipeline ---
async def main():
    prompts = ['RAG', 'agents', 'MCP', 'RAG']  # RAG is duplicate (cache test)
    start = time.time()

    results = await asyncio.gather(*[call_llm(p) for p in prompts])
    print(f'4 calls in {time.time() - start:.1f}s\n')

    for r in results:
        print(f'[{r.confidence:.0%}, {r.tokens}tok] ', end='')
        for w in stream(r.answer):
            print(w, end='', flush=True)
        print()

await main()

</details>

---

## 🎉 Done!

You've mastered all 7 Python skills for GenAI:
- **Async/Await** — parallel API calls
- **Generators** — token streaming
- **Type Hints + Pydantic** — LLM output validation
- **Decorators** — retry, cache, timer
- **Error Handling** — rate limits, fallback chains
- **Production Pattern** — all skills combined

No other GenAI course teaches this integrated set. Next: **Lesson 0.2 Math for GenAI.**